# MLSM Lab 1 — Entrenamiento interactivo

Selecciona el dataset y el algoritmo, compara todas las configuraciones mediante validación cruzada y, al final, elige una fila para mostrar su matriz de confusión. La matriz se calcula con predicciones out-of-fold, no con predicciones sobre los datos usados para ajustar cada fold.

**Requisitos:** ejecutar desde Jupyter con los CSV en `Lab1/data/` (o `data/` si el notebook está dentro de `Lab1`).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict

# This notebook contains the model definitions as well, so it does not depend on a .py file.
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


def _genreall_models(n_features):
    selector_options = [k for k in [80, 100, 130, 150, 180, 200] if k <= n_features]
    selector_options.append("all")
    common = [
        ("imputer", SimpleImputer(strategy="median")),
        ("variance", VarianceThreshold(threshold=0.0)),
    ]
    return {
        "Logistic Regression": (
            Pipeline(common + [("scaler", StandardScaler()), ("selector", SelectKBest(score_func=f_classif)),
                               ("model", LogisticRegression(max_iter=5000, solver="lbfgs"))]),
            {"selector__k": selector_options, "model__C": [0.03, 0.1, 0.2, 0.3, 0.5, 1.0]},
        ),
        "k-NN": (
            Pipeline(common + [("scaler", StandardScaler()), ("selector", SelectKBest(score_func=f_classif)),
                               ("model", KNeighborsClassifier())]),
            {"selector__k": selector_options, "model__n_neighbors": [5, 7, 9, 11, 15],
             "model__weights": ["uniform", "distance"], "model__p": [1, 2]},
        ),
        "Decision Tree": (
            Pipeline(common + [("selector", SelectKBest(score_func=f_classif)),
                               ("model", DecisionTreeClassifier(random_state=42))]),
            {"selector__k": selector_options, "model__max_depth": [5, 10, 15],
             "model__min_samples_leaf": [5, 10], "model__min_samples_split": [2, 10]},
        ),
        "Artificial Neural Network": (
            Pipeline(common + [("scaler", StandardScaler()), ("selector", SelectKBest(score_func=f_classif)),
                               ("model", MLPClassifier(solver="adam", learning_rate_init=0.001, max_iter=700,
                                                       random_state=42, early_stopping=True, validation_fraction=0.1,
                                                       n_iter_no_change=15, tol=1e-3))]),
            {"selector__k": selector_options, "model__hidden_layer_sizes": [(50,), (100,), (100, 50)],
             "model__alpha": [0.001, 0.01, 0.1]},
        ),
    }


def _features_30_sec_models():
    return {
        "Logistic Regression": (
            Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
                      ("model", LogisticRegression(max_iter=5000, solver="lbfgs"))]),
            {"model__C": [0.01, 0.03, 0.1, 0.3, 1.0]},
        ),
        "k-NN": (
            Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
                      ("model", KNeighborsClassifier())]),
            {"model__n_neighbors": [3, 5, 7, 9, 13, 17], "model__weights": ["uniform", "distance"],
             "model__p": [1, 2]},
        ),
        "Decision Tree": (
            Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("model", DecisionTreeClassifier(random_state=42))]),
            {"model__max_depth": [5, 10, 15], "model__min_samples_split": [5, 10],
             "model__min_samples_leaf": [5, 10]},
        ),
        "Artificial Neural Network": (
            Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
                      ("model", MLPClassifier(max_iter=500, random_state=42, early_stopping=False,
                                              n_iter_no_change=15, tol=1e-3))]),
            {"model__hidden_layer_sizes": [(30,), (50,), (100,), (50, 25)],
             "model__alpha": [0.01, 0.1, 1.0]},
        ),
    }


def _features_3_sec_models():
    return {
        "Logistic Regression": (
            Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
                      ("model", LogisticRegression(max_iter=5000, solver="lbfgs"))]),
            {"model__C": [0.03, 0.1, 0.3, 1.0, 3.0]},
        ),
        "k-NN": (
            Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
                      ("model", KNeighborsClassifier())]),
            {"model__n_neighbors": [5, 9, 15, 25], "model__weights": ["distance"], "model__p": [1, 2]},
        ),
        "Decision Tree": (
            Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("model", DecisionTreeClassifier(random_state=42))]),
            {"model__max_depth": [10, 20, None], "model__min_samples_split": [5, 10, 20],
             "model__min_samples_leaf": [2, 5, 10]},
        ),
        "Artificial Neural Network": (
            Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
                      ("model", MLPClassifier(solver="adam", learning_rate_init=0.001, max_iter=500,
                                              random_state=42, early_stopping=True, validation_fraction=0.1,
                                              n_iter_no_change=15, tol=1e-3))]),
            {"model__hidden_layer_sizes": [(50,), (100,), (100, 50)], "model__alpha": [0.001, 0.01, 0.1]},
        ),
    }


def get_models(dataset, n_features):
    if dataset == "GenreAll":
        return _genreall_models(n_features)
    if dataset == "features-30-sec":
        return _features_30_sec_models()
    if dataset == "features-3-sec":
        return _features_3_sec_models()
    raise ValueError(f"Unknown dataset: {dataset}")


BASE_DIR = Path.cwd()
# If the notebook is opened from the repository root, use Lab1/data; if opened from Lab1, use data.
DATA_DIR = BASE_DIR / "data" if (BASE_DIR / "data").exists() else BASE_DIR / "Lab1" / "data"

DATASETS = {
    "1": ("GenreAll", "GenreAll.csv", "genre"),
    "2": ("features-30-sec", "features_30_sec.csv", "label"),
    "3": ("features-3-sec", "features_3_sec.csv", "label"),
}
MODEL_OPTIONS = {"l": "Logistic Regression", "k": "k-NN", "d": "Decision Tree", "a": "Artificial Neural Network"}

def choose_dataset():
    print("\nSelect the dataset:\n  1 -> GenreAll\n  2 -> features-30-sec\n  3 -> features-3-sec")
    while True:
        choice = input("Dataset [1/2/3]: ").strip().lower()
        if choice in DATASETS:
            return DATASETS[choice]
        print("Invalid option. Enter 1, 2 or 3.")

def choose_models(models):
    print("\nSelect the model(s) to train:\n  l -> Logistic Regression\n  k -> k-NN\n  d -> Decision Tree\n  a -> Artificial Neural Network\n  all -> Train all models")
    while True:
        choice = input("Model [l/k/d/a/all]: ").strip().lower()
        if choice == "all":
            return models.copy()
        if choice in MODEL_OPTIONS:
            name = MODEL_OPTIONS[choice]
            if name in models:
                return {name: models[name]}
            print(f"{name} is not defined for this dataset.")
        else:
            print("Invalid option. Enter l, k, d, a or all.")

def load_dataset(csv_filename, target_column):
    csv_path = DATA_DIR / csv_filename
    if not csv_path.exists():
        raise FileNotFoundError(f"Dataset not found: {csv_path}. Check that CSV files are in Lab1/data/.")
    df = pd.read_csv(csv_path, skipinitialspace=True).dropna(subset=[target_column])
    y = df[target_column].copy()
    X = df.drop(columns=[target_column]).copy()
    text_columns = X.select_dtypes(include=["object", "string"]).columns
    for column in text_columns:
        values = X[column].dropna().unique()
        if len(values) == 2:
            X[column] = X[column].map({values[0]: 0, values[1]: 1})
    X = X.select_dtypes(include=np.number)
    if X.empty:
        raise ValueError(f"No numeric features found in {csv_filename}.")
    print(f"\nDataset: {csv_filename}\nSamples: {len(df)}\nNumeric features: {X.shape[1]}\n")
    display(y.value_counts().rename("samples per class").to_frame())
    return X, y

def train_models(models, X, y, cv):
    results = []
    for name, (pipeline, param_grid) in models.items():
        print(f"\nTraining: {name}")
        search = GridSearchCV(pipeline, param_grid, scoring="accuracy", cv=cv,
                              return_train_score=True, n_jobs=-1, error_score="raise")
        search.fit(X, y)
        for i, params in enumerate(search.cv_results_["params"]):
            train_acc = search.cv_results_["mean_train_score"][i]
            cv_acc = search.cv_results_["mean_test_score"][i]
            results.append({"Algorithm": name, "Parameters": params, "Train accuracy": train_acc,
                            "CV accuracy": cv_acc, "Overfitting gap": train_acc - cv_acc})
        print(f"Best parameters: {search.best_params_}")
        print(f"Best CV accuracy: {search.best_score_:.4f}")
    return pd.DataFrame(results).sort_values(["CV accuracy", "Overfitting gap"],
                                              ascending=[False, True]).reset_index(drop=True)

def plot_confusion_matrix(row_index, results_df, models, X, y, cv):
    row = results_df.iloc[row_index]
    model_name, params = row["Algorithm"], row["Parameters"]
    print(f"Algorithm: {model_name}\nParameters: {params}\nCV accuracy: {row['CV accuracy']:.4f}")
    selected_model = clone(models[model_name][0]).set_params(**params)
    y_pred = cross_val_predict(selected_model, X, y, cv=cv, n_jobs=-1)
    classes = np.unique(y)
    display = ConfusionMatrixDisplay(confusion_matrix(y, y_pred, labels=classes), display_labels=classes)
    fig, ax = plt.subplots(figsize=(10, 8))
    display.plot(ax=ax, xticks_rotation="vertical", values_format="d", colorbar=False)
    ax.set_title(f"Confusion matrix - {model_name} (result row {row_index})")
    plt.tight_layout()
    plt.show()

# Run the experiment. This uses out-of-fold predictions for the confusion matrix.
dataset_name, csv_filename, target_column = choose_dataset()
X, y = load_dataset(csv_filename, target_column)
available_models = get_models(dataset_name, X.shape[1])
selected_models = choose_models(available_models)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
results_df = train_models(selected_models, X, y, cv)
display(results_df.style.format({"Train accuracy": "{:.4f}", "CV accuracy": "{:.4f}", "Overfitting gap": "{:.4f}"}))
print(f"Total configurations evaluated: {len(results_df)}")

choice = input(f"\nSelect a result row for its confusion matrix [0-{len(results_df)-1}], or q to skip: ").strip().lower()
if choice != "q":
    try:
        row_index = int(choice)
        if 0 <= row_index < len(results_df):
            plot_confusion_matrix(row_index, results_df, selected_models, X, y, cv)
        else:
            print("Row out of range; confusion matrix skipped.")
    except ValueError:
        print("Invalid input; confusion matrix skipped.")
else:
    print("Confusion matrix skipped.")
